<a href="https://colab.research.google.com/github/rkfussell/Physics_Affinity/blob/main/physicsAff_useful.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

## **Set up**

This code section imports necessary packages and the data from Google Drive.

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
import re

import pandas as pd
import numpy as np
from tqdm import tqdm
from transformers import AutoTokenizer, AutoModelForCausalLM
from huggingface_hub import login
login()

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:104: UserWarning: 
Error while fetching `HF_TOKEN` secret value from your vault: 'Requesting secret HF_TOKEN timed out. Secrets can only be fetched when running from the Colab UI.'.
You are not authenticated with the Hugging Face Hub in this notebook.
If the error persists, please let us know by opening an issue on GitHub (https://github.com/huggingface/huggingface_hub/issues/new).
  warnings.warn(


In [ ]:
from google.colab import drive
drive.mount('/content/drive', force_remount=True)


confidence = pd.read_excel("/content/drive/My Drive/Colab Notebooks/2024_fall_Cornell_Phys1101and2207_physaff_Rnd2_CollectiveCoding.xlsx",sheet_name="Confidence")
interest = pd.read_excel("/content/drive/My Drive/Colab Notebooks/2024_fall_Cornell_Phys1101and2207_physaff_Rnd2_CollectiveCoding.xlsx",sheet_name="Interest & Use")
confidence_codebook = pd.read_excel("/content/drive/My Drive/Colab Notebooks/2024_fall_Cornell_Phys1101and2207_physaff_Rnd2_CollectiveCoding.xlsx",sheet_name="Codebook-Confidence")
interest_codebook = pd.read_excel("/content/drive/My Drive/Colab Notebooks/2024_fall_Cornell_Phys1101and2207_physaff_Rnd2_CollectiveCoding.xlsx",sheet_name="Codebook-InterestandUse")

Mounted at /content/drive


**Run this cell to view the data that has been loaded in.**

In [ ]:
interest.head()

,Unnamed: 0,course,studentID,In a few (short) sentences: Do you expect to find learning\nphysics to be interesting and useful? Why or why not?,Interesting,Not interesting,Useful,Not Useful,Enjoyable,Applicable in Bio,...,math,how/why things work,daily life,essential knowledge,ability,required/MCAT,Unnamed: 18,Unnamed: 19,Unnamed: 20,Unnamed: 21
0,NaN,PHYS 2207,85ef67d0-649f-4e26-9545-1382bd4086a3,"I expect learning physics to be useful, but no...",NaN,x,x,NaN,NaN,NaN,...,x,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,NaN,PHYS 2207,2d8769ee-10f4-4635-8a2e-f8e6af40abd3,"I expect physics to be interesting, because I ...",x,NaN,x,NaN,x,NaN,...,NaN,x,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,NaN,PHYS 2207,b6b37f0b-dad5-4d8c-acbe-4a72a693eb6e,I think physics is both interesting and useful...,x,NaN,x,NaN,NaN,x,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,NaN,PHYS 2207,f8f33e36-9c73-4007-a43b-3fcf1122f2ab,I do not expect learning physics to be interes...,NaN,x,NaN,x,NaN,NaN,...,NaN,x,NaN,NaN,NaN,NaN,not enjoyable,NaN,NaN,NaN
4,NaN,PHYS 2207,dbb358dd-c4e8-4714-84d9-a32e477edfd8,I think it will be interesting but I do not th...,x,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


**This line changes the question text to ``questionStem`` which we can use as a variable to identify the right column later.**

In [ ]:

interest = interest.rename(columns={"In a few (short) sentences: Do you expect to find learning\nphysics to be interesting and useful? Why or why not?":"questionStem"})
interest = interest.fillna(0)
interest = interest.replace('x',1)

/tmp/ipykernel_3620/2372801374.py:3: FutureWarning: Downcasting behavior in `replace` is deprecated and will be removed in a future version. To retain the old behavior, explicitly call `result.infer_objects(copy=False)`. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  interest = interest.replace('x',1)


In [ ]:
interest

,Unnamed: 0,course,studentID,questionStem,Interesting,Not interesting,Useful,Not Useful,Enjoyable,Applicable in Bio,...,math,how/why things work,daily life,essential knowledge,ability,required/MCAT,Unnamed: 18,Unnamed: 19,Unnamed: 20,Unnamed: 21
0,0.0,PHYS 2207,85ef67d0-649f-4e26-9545-1382bd4086a3,"I expect learning physics to be useful, but no...",0,1,1,0,0,0,...,1,0,0,0,0,0,0,0,0,0.0
1,0.0,PHYS 2207,2d8769ee-10f4-4635-8a2e-f8e6af40abd3,"I expect physics to be interesting, because I ...",1,0,1,0,1,0,...,0,1,0,0,0,0,0,0,0,0.0
2,0.0,PHYS 2207,b6b37f0b-dad5-4d8c-acbe-4a72a693eb6e,I think physics is both interesting and useful...,1,0,1,0,0,1,...,0,0,0,0,0,0,0,0,0,0.0
3,0.0,PHYS 2207,f8f33e36-9c73-4007-a43b-3fcf1122f2ab,I do not expect learning physics to be interes...,0,1,0,1,0,0,...,0,1,0,0,0,0,not enjoyable,0,0,0.0
4,0.0,PHYS 2207,dbb358dd-c4e8-4714-84d9-a32e477edfd8,I think it will be interesting but I do not th...,1,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
667,0.0,PHYS 2207,d919df7f-ae2e-486f-b563-8d7780fbc994,I guess the parts of physics where I can easil...,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0.0
668,0.0,PHYS 2207,b4e7cbfc-8d41-4e31-b841-89b8dbf1d0ba,I think it will be since it can be applied to ...,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0.0
669,0.0,PHYS 1101,35f501b0-2ac2-4666-bd69-aa71af516978,"Yes, I want to do biotech so I think it will c...",0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0.0
670,0.0,PHYS 1101,882ec822-d244-482f-9fea-b2abba2f1643,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0.0


In [ ]:
#load model
model_name = "google/gemma-3-4b-it"
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForCausalLM.from_pretrained(
    model_name,
    output_hidden_states=True,
    dtype=torch.float32,
    device_map="auto"
)

config.json:   0%|          | 0.00/855 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/1.16M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/33.4M [00:00<?, ?B/s]

added_tokens.json:   0%|          | 0.00/35.0 [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/662 [00:00<?, ?B/s]

model.safetensors.index.json:   0%|          | 0.00/90.6k [00:00<?, ?B/s]

Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

The following generation flags are not valid and may be ignored: ['output_hidden_states']. Set `TRANSFORMERS_VERBOSITY=info` for more details.


Loading weights:   0%|          | 0/883 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/215 [00:00<?, ?B/s]

## **Prompt Set Up**

In this section, we'll define our prompt and the subsets of data we want to use to test it.**bold text**

In [ ]:

prompt = "Help me analyze some student statements. In each statement, decide if the student says physics is Useful, Not Useful, or you Can't Tell. If the student statement does not include the word ``useful'', categorize it as Can't Tell. Here are some examples:"

prompt+= "\n \n Example 1: ``I expect learning physics to be useful, but not very interesting. It expect that it would take a lot of problem solving and math rather than exploring theoretic and conceptual ideas.''"

prompt+= "\n Answer: **Useful** \n Explanation: The student says 'I expect learning physics to be useful', which is a positive statement about its use."

prompt+= "\n \n Example 2: ``I do not expect to find it to be useful because I am a biological sciences major, but perhaps it will be interesting.''"

prompt+= "\n Answer: **Not Useful** \n Explanation: The student says they ``don't expect to find it useful', which is a negative statement about its use."

prompt+= "\n \n Example 3: ``Yes, I think that physics is important to learn however I'm not sure if it will be interesting. I hope it is.''"

prompt+= "\n Answer: **Can't Tell** \n Explanation: The student does not say the word ``useful'' so we can't tell what they think about its use."

prompt+= "\n \n Example 4: I think that learning physics will be interesting but I am not sure how useful it'll be in the career that i want to pursue.''"

prompt+= "\n Answer: **Can't Tell** \n Explanation: The student says they are ``not sure'' which is not a positive nor a negative statement about its use."

prompt+= "\n \n Example 5: ``Not really; I have never been good at it.''"

prompt+= "\n Answer: **Can't Tell** \n Explanation: The student talks about their ability to do physics, not its use."

prompt+= "\n \n Here is another student statement. Tell me if the student says physics is Useful, physics is Not Useful, or you Can't Tell. Provide your answer first and then explain your reasoning. \n"













**Define subsets of the coded data so we can quickly check if Gemma has coded correctly.**

In [ ]:
useful = interest.loc[interest["Useful"] == 1]
notUseful = interest.loc[interest["Not Useful"] == 1]
cantTell = interest.loc[(interest["Useful"] == 0) & (interest["Not Useful"] == 0)]

## **Testing: Useful**

This code applies the prompt just to the useful subset, so we want Gemma to code everything as ``useful``.

In [ ]:
promptWithStatement=prompt + "``" + useful.iloc[3]['questionStem'] + "''"
print(promptWithStatement)

Help me analyze some student statements. In each statement, decide if the student says physics is Useful, Not Useful, or you Can't Tell. If the student doesn't talk about use explicitly, code it as Can't Tell. Here are some examples:
 
 Example 1: ``I expect learning physics to be useful, but not very interesting. It expect that it would take a lot of problem solving and math rather than exploring theoretic and conceptual ideas.''
 Answer: **Useful** 
 Explanation: The student says 'I expect learning physics to be useful', which is a positive statement about its use.
 
 Example 2: ``I do not expect to find it to be useful because I am a biological sciences major, but perhaps it will be interesting''
 Answer: **Not Useful** 
 Explanation: Even though the student says they find physics interesting, they also say they don't expect to find it useful', which is a negative statement about its use.
 
 Example 3: ``Yes, I think that physics is important to learn however I'm not sure if it will

In [ ]:
base=prompt

count_useful, count_notUseful, count_noCode, count_fuckinGemma =0,0,0,0
with torch.no_grad():
  for i in range(0,144):

      promptWithStatement=base + "``" + useful.iloc[i]['questionStem'] + "''"
      inputs = tokenizer(promptWithStatement, return_tensors="pt").to(model.device)
      # Run inference
      outputs = model.generate(**inputs, max_new_tokens=10)
      # Decode the generated response
      ans = tokenizer.batch_decode(outputs, skip_special_tokens=True)[0]
      # Define the first bit to get just Gemma's categorization
      ans_split = ans.split(useful.iloc[i]['questionStem'])[-1]
      gemmasFeelings = re.findall(r'\*\*(.*?)\*\*',ans_split)

      print("\n")
      print(i)
      # print("The full output is:")
      # print(ans)
      print("The sentence was: ")
      print(useful.iloc[i]['questionStem'])

      if len(gemmasFeelings) == 0:
          count_fuckinGemma+=1
          print("Gemma feels that it is: ")
          print(ans_split)
          continue
      else:
          print("The stripped answer is: ")
          print(gemmasFeelings[0])

      # print("Gemma's feelings 1st element: ")
      # print(gemmasFeelings[0])

      if "Useful" == gemmasFeelings[0]:
          count_useful+=1
      elif "Not Useful" == gemmasFeelings[0]:
          count_notUseful+=1
      elif "Can't Tell" == gemmasFeelings[0]:
          count_noCode+=1
      else:
          print("The sentence was: ")
          print(useful.iloc[i]['questionStem'])
          print("Gemma feels that it is: ")
          print(ans_split)

print(count_useful)
print(count_notUseful)
print(count_noCode)
print(count_fuckinGemma)





0
The sentence was: 
I expect learning physics to be useful, but not very interesting. It expect that it would take a lot of problem solving and math rather than exploring theoretic and conceptual ideas
The stripped answer is: 
Useful


1
The sentence was: 
I expect physics to be interesting, because I like to learn about how we can predict trajectories of a ball and perform physics labs. I think it will be very useful to better understanding our environment and how the world works.
The stripped answer is: 
Useful


2
The sentence was: 
I think physics is both interesting and useful. It is very applicable to the real world in the healthcare sector particularly. If also provides explanation for certain biological processes.
The stripped answer is: 
Useful


3
The sentence was: 
Learning physics will be useful to me. I am sure that understanding physics concepts will help me get insights about biological systems,  where passion and interests are.
The stripped answer is: 
Useful


4
The

## **Testing: Not Useful**

This code tests the not useful subset, again so we can make sure the prompt is working well.

In [ ]:
print(prompt)

Help me analyze some student statements. In each statement, decide if the student says physics is Useful, Not Useful, or you Can't Tell. If the student doesn't talk about use explicitly, code it as Can't Tell. Here are some examples:
 
 Example 1: ``I expect learning physics to be useful, but not very interesting. It expect that it would take a lot of problem solving and math rather than exploring theoretic and conceptual ideas.''
 Answer: **Useful** 
 Explanation: The student says 'I expect learning physics to be useful', which is a positive statement about its use.
 
 Example 2: ``I do not expect to find it to be useful because I am a biological sciences major, but perhaps it will be interesting''
 Answer: **Not Useful** 
 Explanation: Even though the student says they find physics interesting, they also say they don't expect to find it useful', which is a negative statement about its use.
 
 Example 3: ``Yes, I think that physics is important to learn however I'm not sure if it will

In [ ]:
base=prompt

count_useful, count_notUseful, count_noCode, count_fuckinGemma =0,0,0,0
with torch.no_grad():
  for i in range(0,17):

      promptWithStatement=base + "``" + notUseful.iloc[i]['questionStem'] + "''"
      inputs = tokenizer(promptWithStatement, return_tensors="pt").to(model.device)
      # Run inference
      outputs = model.generate(**inputs, max_new_tokens=20)
      # Decode the generated response
      ans = tokenizer.batch_decode(outputs, skip_special_tokens=True)[0]
      # Define the first bit to get just Gemma's categorization
      ans_split = ans.split(notUseful.iloc[i]['questionStem'])[-1]
      gemmasFeelings = re.findall(r'\*\*(.*?)\*\*',ans_split)

      print("\n")
      print(i)
      # print("The full output is:")
      # print(ans)
      print("The sentence was: ")
      print(notUseful.iloc[i]['questionStem'])

      if len(gemmasFeelings) == 0:
          count_fuckinGemma+=1
          print("Gemma feels that it is: ")
          print(ans_split)
          continue
      else:
          print("The stripped answer is: ")
          print(gemmasFeelings[0])

      # print("Gemma's feelings 1st element: ")
      # print(gemmasFeelings[0])

      if "Useful" == gemmasFeelings[0]:
          count_useful+=1
      elif "Not Useful" == gemmasFeelings[0]:
          count_notUseful+=1
      elif "Can't Tell" == gemmasFeelings[0]:
          count_noCode+=1
      else:
          print("The sentence was: ")
          print(notUseful.iloc[i]['questionStem'])
          print("Gemma feels that it is: ")
          print(ans_split)

print(count_useful)
print(count_notUseful)
print(count_noCode)
print(count_fuckinGemma)





0
The sentence was: 
I do not expect learning physics to be interesting and useful. I don't really understand the point of knowing a bunch of information about why/how things move, because it doesn't really pertain to my major. I also don't enjoy physics as a subject, so I don't see myself applying it in my life during/after the class. 
The stripped answer is: 
Not Useful


1
The sentence was: 
I expect it to be interesting maybe not as useful. I want to be a surgeon and I don't think physics will come up a lot.
The stripped answer is: 
Not Useful


2
The sentence was: 
I think that learning physics will be interesting but I am not sure how useful it'll be in the career that i want to pursue.
The stripped answer is: 
Can't Tell


3
The sentence was: 
I think it will be kind of interesting but not really useful because in this introductory course I doubt if I can learn anything really applicable in my biology study.
The stripped answer is: 
Not Useful


4
The sentence was: 
I don't ne

## **Testing: Can't Tell / Neither**

This code tests the not useful subset, again so we can make sure the prompt is working well.

In [ ]:
promptTakeTwo = "I asked students if they thought physics was interesting and/or useful. Help me analyze their responses to figure out which students think its USEFUL and how many think it is NOT USEFUL."
promptTakeTwo+= "\n For each student statement, first report if the student answers the question about USE."
promptTakeTwo+= "\n Only if the statement contains the word USE or USEFUL, decide if the student thinks physics is USEFUL, NOT USEFUL, or you CAN'T TELL."
promptTakeTwo+= "\n If the statement does not contain the word USE or USEFUL, you must label the statement as CAN'T TELL. Here are some examples:"

promptTakeTwo+= "\n \n Example 1: I expect learning physics to be useful, but not very interesting. It expect that it would take a lot of problem solving and math rather than exploring theoretic and conceptual ideas."

promptTakeTwo+= "\n Answer: **USEFUL** Explanation: The statement contains the word ``USEFUL'' and the student is a positive about its use."

promptTakeTwo+= "\n \n Example 2: I do not expect to find it to be useful because I am a biological sciences major, but perhaps it will be interesting."

promptTakeTwo+= "\n Answer: **NOT USEFUL** Explanation: The statement contains the word ``USEFUL'' and the student is a negative about its use."

promptTakeTwo+= "\n \n Example 3: Not really; I have never been good at it."

promptTakeTwo+= "\n Answer: **CAN'T TELL** Explanation: The statement does not contain the words ``USEFUL'' or ``USE''."

promptTakeTwo+= "\n \n Example 4: I think that learning physics will be interesting but I am not sure how useful it'll be in the career that i want to pursue."

promptTakeTwo+= "\n Answer: **CAN'T TELL** Explanation: The statement contains the word ``USEFUL'', but also says that the they are ``not sure'' which is not only positive or only negative."

promptTakeTwo+= "\n \n Here is another student statement. Tell me if the statement should be categorized as USEFUL, NOT USEFUL, or CAN'T TELL. Provide your answer first and then explain your reasoning. \n"

In [ ]:
print(promptTakeTwo)

I asked students if they thought physics was interesting and/or useful. Help me analyze their responses to figure out which students think its USEFUL and how many think it is NOT USEFUL.
 For each student statement, first report if the student answers the question about USE.
 Only if the statement contains the word USE or USEFUL, decide if the student thinks physics is USEFUL, NOT USEFUL, or you CAN'T TELL.
 If the statement does not contain the word USE or USEFUL, you must label the statement as CAN'T TELL. Here are some examples:
 
 Example 1: I expect learning physics to be useful, but not very interesting. It expect that it would take a lot of problem solving and math rather than exploring theoretic and conceptual ideas.
 Answer: **USEFUL** Explanation: The statement contains the word ``USEFUL'' and the student is a positive about its use.
 
 Example 2: I do not expect to find it to be useful because I am a biological sciences major, but perhaps it will be interesting.
 Answer: **N

In [ ]:
base=promptTakeTwo

count_useful, count_notUseful, count_noCode, count_fuckinGemma =0,0,0,0
with torch.no_grad():
  for i in range(0,15):

      promptWithStatement=base + "Statement: ``" + cantTell.iloc[i]['questionStem'] + "''"
      inputs = tokenizer(promptWithStatement, return_tensors="pt").to(model.device)
      # Run inference
      outputs = model.generate(**inputs, max_new_tokens=25)
      # Decode the generated response
      ans = tokenizer.batch_decode(outputs, skip_special_tokens=True)[0]
      # Define the first bit to get just Gemma's categorization
      ans_split = ans.split(cantTell.iloc[i]['questionStem'])[-1]
      gemmasFeelings = re.findall(r'\*\*(.*?)\*\*',ans_split)

      print("\n")
      print(i)
      print("The full output is:")
      print(ans)
      # print("The sentence was: ")
      # print(cantTell.iloc[i]['questionStem'])

      if len(gemmasFeelings) == 0:
          count_fuckinGemma+=1
          print("Gemma feels that it is: ")
          print(ans_split)
          continue
      else:
          print("The stripped answer is: ")
          print(gemmasFeelings[0])

      # print("Gemma's feelings 1st element: ")
      # print(gemmasFeelings[0])

      if "USEFUL" == gemmasFeelings[0]:
          count_useful+=1
      elif "NOT USEFUL" == gemmasFeelings[0]:
          count_notUseful+=1
      elif "CAN'T TELL" == gemmasFeelings[0]:
          count_noCode+=1
      else:
          print("The sentence was: ")
          print(cantTell.iloc[i]['questionStem'])
          print("Gemma feels that it is: ")
          print(ans_split)

print(count_useful)
print(count_notUseful)
print(count_noCode)
print(count_fuckinGemma)





0
The full output is:
I asked students if they thought physics was interesting and/or useful. Help me analyze their responses to figure out which students think its USEFUL and how many think it is NOT USEFUL.
 For each student statement, first report if the student answers the question about USE.
 Only if the statement contains the word USE or USEFUL, decide if the student thinks physics is USEFUL, NOT USEFUL, or you CAN'T TELL.
 If the statement does not contain the word USE or USEFUL, you must label the statement as CAN'T TELL. Here are some examples:
 
 Example 1: I expect learning physics to be useful, but not very interesting. It expect that it would take a lot of problem solving and math rather than exploring theoretic and conceptual ideas.
 Answer: **USEFUL** Explanation: The statement contains the word ``USEFUL'' and the student is a positive about its use.
 
 Example 2: I do not expect to find it to be useful because I am a biological sciences major, but perhaps it will be i

## **Testing: Entire Dataset that has been coded for direct comparison**

**Make a blank dataframe to fill in with the results from this run, so we can compare it to the coded data**

In [ ]:
gemmasCodes = interest[['studentID', 'Useful','Not Useful']].copy()

In [ ]:
gemmasCodes['Useful'] = 0
gemmasCodes['Not Useful'] = 0

gemmasCodes.head()

,studentID,Useful,Not Useful
0,85ef67d0-649f-4e26-9545-1382bd4086a3,0,0
1,2d8769ee-10f4-4635-8a2e-f8e6af40abd3,0,0
2,b6b37f0b-dad5-4d8c-acbe-4a72a693eb6e,0,0
3,f8f33e36-9c73-4007-a43b-3fcf1122f2ab,0,0
4,dbb358dd-c4e8-4714-84d9-a32e477edfd8,0,0


In [ ]:
base=prompt

count_useful, count_notUseful, count_noCode, count_fuckinGemma =0,0,0,0
with torch.no_grad():
  for i in range(0,100):

      promptWithStatement=base + "``" + interest.iloc[i]['questionStem'] + "''"
      inputs = tokenizer(promptWithStatement, return_tensors="pt").to(model.device)
      # Run inference
      outputs = model.generate(**inputs, max_new_tokens=20)
      # Decode the generated response
      ans = tokenizer.batch_decode(outputs, skip_special_tokens=True)[0]
      # Define the first bit to get just Gemma's categorization
      ans_split = ans.split(interest.iloc[i]['questionStem'])[-1]
      gemmasFeelings = re.findall(r'\*\*(.*?)\*\*',ans_split)

      print("\n")
      print(i)
      # print("The full output is:")
      # print(ans)
      print("The sentence was: ")
      print(interest.iloc[i]['questionStem'])

      if len(gemmasFeelings) == 0:
          count_fuckinGemma+=1
          print("Gemma feels that it is: ")
          print(ans_split)
          continue
      else:
          print("The stripped answer is: ")
          print(gemmasFeelings[0])

      # print("Gemma's feelings 1st element: ")
      # print(gemmasFeelings[0])

      if "Useful" == gemmasFeelings[0]:
          count_useful+=1
          gemmasCodes.at[i,'Useful'] = 1
      elif "Not Useful" == gemmasFeelings[0]:
          count_notUseful+=1
          gemmasCodes.at[i,'Not Useful'] = 1
      elif "Can't Tell" == gemmasFeelings[0]:
          count_noCode+=1
      else:
          # print("The sentence was: ")
          # print(interest.iloc[i]['questionStem'])
          # print("Gemma feels that it is: ")
          # print(ans_split)
          continue

print(count_useful)
print(count_notUseful)
print(count_noCode)
print(count_fuckinGemma)



0
The sentence was: 
I expect learning physics to be useful, but not very interesting. It expect that it would take a lot of problem solving and math rather than exploring theoretic and conceptual ideas
The stripped answer is: 
Useful


1
The sentence was: 
I expect physics to be interesting, because I like to learn about how we can predict trajectories of a ball and perform physics labs. I think it will be very useful to better understanding our environment and how the world works.
The stripped answer is: 
Useful


2
The sentence was: 
I think physics is both interesting and useful. It is very applicable to the real world in the healthcare sector particularly. If also provides explanation for certain biological processes.
The stripped answer is: 
Useful


3
The sentence was: 
I do not expect learning physics to be interesting and useful. I don't really understand the point of knowing a bunch of information about why/how things move, because it doesn't really pertain to my major. I a

In [ ]:
gemmasCodes

,studentID,Useful,Not Useful
0,85ef67d0-649f-4e26-9545-1382bd4086a3,1,0
1,2d8769ee-10f4-4635-8a2e-f8e6af40abd3,1,0
2,b6b37f0b-dad5-4d8c-acbe-4a72a693eb6e,1,0
3,f8f33e36-9c73-4007-a43b-3fcf1122f2ab,1,1
4,dbb358dd-c4e8-4714-84d9-a32e477edfd8,0,1
...,...,...,...
667,d919df7f-ae2e-486f-b563-8d7780fbc994,0,0
668,b4e7cbfc-8d41-4e31-b841-89b8dbf1d0ba,0,0
669,35f501b0-2ac2-4666-bd69-aa71af516978,0,0
670,882ec822-d244-482f-9fea-b2abba2f1643,0,0


In [ ]:
gemmasCodes.to_csv('gemmasCodes_Interest_First100.csv', index=False)